# Build the eddy-centre stratification cache

This notebook calculates environmental buoyancy frequency and writes one row per existing eddy-day. It **does not calculate or modify tilt**. `xroms.potential_density(..., z=0)` supplies surface-referenced potential density, then $N^2=-g\rho_0^{-1}\partial\sigma_0/\partial z$ is evaluated down each requested column. Using potential rather than pressure-dependent in-situ density prevents adiabatic compression from being misidentified as stratification. The cache contains fixed-depth mean, integral, and maximum $N^2$ over 0–200 and 0–500 m, plus pycnocline-following mean/maximum $N^2$, pycnocline depth, and density-threshold mixed-layer depth.

The build is restartable and parallel by 30-day ROMS file. Within each file, all eddy centres required on a timestep are selected together and computed once. Completed files are stored under a partition folder containing a hash of the scientific settings, so partitions cannot be silently reused after changing the method, depths, MLD threshold, or pycnocline window.

Run this once on Katana before notebook 02. Inspect the model variable names and the sign/range checks before accepting the cache.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


In [ ]:
MODEL_ROOT = Path("/srv/scratch/z3533156/26year_BRAN2020")
N2_CACHE = mech.DEFAULT_N2_CACHE_PATH
WORKERS = 7              # Start with 4-8; increase only after checking memory.
POINT_BATCH_SIZE = 128  # Vectorized eddy columns per backend read.

required_model_columns = ["Eddy", "Day", "ic", "jc", "xc", "yc", "Rc", "q11", "q12", "q22"]
df[required_model_columns].isna().mean().rename("missing_fraction")


In [ ]:
# One process per model file; completed partitions are reused after interruption.
n2_cache = mech.build_n2_cache_xroms(
    df,
    MODEL_ROOT,
    N2_CACHE,
    depths=(200, 500),
    workers=WORKERS,
    point_batch_size=POINT_BATCH_SIZE,
    grid_path=paths.grid,
    z_r_path=paths.z_r,
    skip_existing=True,
)
n2_cache.head()


In [ ]:
core_cols = ["N2_200m_core_s2", "N2_500m_core_s2"]
centre_cols = ["N2_200m_centre_s2", "N2_500m_centre_s2"]
display(n2_cache[core_cols + centre_cols].describe(percentiles=[.01, .05, .5, .95, .99]))
assert not n2_cache.duplicated(["Eddy", "Day"]).any()
for col in core_cols:
    print(col, "positive fraction:", (n2_cache[col] > 0).mean(), "missing:", n2_cache[col].isna().mean())
    if (n2_cache[col] > 0).mean() < 0.8:
        raise ValueError(f"{col} failed sign QC; check vertical-coordinate alignment.")

metric_cols = ["N2_200m_integral_m_s2_core", "N2_500m_integral_m_s2_core",
               "N2_200m_max_s2_core", "N2_500m_max_s2_core",
               "N2_pycnocline_mean_s2_core", "N2_pycnocline_max_s2_core",
               "pycnocline_depth_m_core", "MLD_density_m_core"]
display(n2_cache[metric_cols].describe(percentiles=[.01, .05, .5, .95, .99]))
display(n2_cache[["N2_core_cells", "N2_200m_core_valid_fraction", "N2_500m_core_valid_fraction"]].describe())
display(n2_cache[["N2_cache_version", "N2_cache_signature", "N2_density_method"]].drop_duplicates())
n2_cache[core_cols].plot.hist(bins=80, logy=True, alpha=.55)
plt.xlabel(r"$N^2$ (s$^{-2}$)")
plt.title("Eddy-core depth-mean stratification QC")
plt.show()


## Acceptance checks

- Confirm `temp`, `salt`, ROMS vertical coordinates, `ic → xi_rho`, and `jc → eta_rho` on several manually selected profiles.
- Investigate negative or extreme values rather than silently clipping them. The depth means retain resolved negative values.
- Compare several cached columns against direct plots of `N2(z)`.
- Record the installed `xroms` version with the final results.


In [ ]:
import xroms
print("xroms", xroms.__version__)
print("cache", N2_CACHE)
